================================================================
FACR — Complete Research Pipeline
Factuality-Aware Contrastive Regularization
================================================================
STRUCTURE:
  CELL 1  — Installs
  CELL 2  — Imports, model loading
  CELL 3  — Core framework functions (no hardcoded layers)
  CELL 4  — Dataset loaders (TriviaQA, HaluEval, TruthfulQA, NQ-Open)
  CELL 5  — EXPERIMENT A: full sequence, all datasets
  CELL 6  — EXPERIMENT B: early detection sweep
  CELL 7  — EXPERIMENT C: cross-dataset transfer
  CELL 8  — Figures (4 publication-ready plots)
  CELL 9  — INTROSPECTION: conclusions + decision for next step
  CELL 10 — EXPERIMENT D: contrastive loss (decoupled from SFT)
  CELL 11 — EXPERIMENT E: before vs after contrastive geometry
  CELL 12 — Final results table (paper-ready)

KEY DESIGN PRINCIPLE:
  No hardcoded layer numbers anywhere.
  find_and_evaluate() automatically finds L* for any
  model + dataset + cutoff combination using a 3-way split:
    20% → layer selection   (find L* via quick sweep)
    60% → probe training    (train at L*)
    20% → evaluation        (final unbiased AUROC)
================================================================

In [ ]:
# ── CELL 1 — Installs ────────────────────────────────────────────
!pip install transformers datasets peft accelerate bitsandbytes tqdm scikit-learn matplotlib -q

── CELL 2 — Imports & model loading ────────────────────────────

In [ ]:
import os, json, re, string, random
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.decomposition import PCA
from torch.utils.data import Dataset as TorchDataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
SAVE_DIR  = '/content/drive/MyDrive/facr_v2'
STEP1_DIR = f'{SAVE_DIR}/step1_results'
STEP2_DIR = f'{SAVE_DIR}/step2_results'
os.makedirs(STEP1_DIR, exist_ok=True)
os.makedirs(STEP2_DIR, exist_ok=True)

In [ ]:
from huggingface_hub import login
login("<REDACTED>")   # <-- replace with your token

In [ ]:
MODEL_ID    = "meta-llama/Llama-3.2-3B-Instruct"
DEVICE      = "cuda"
NUM_LAYERS  = 28     # Llama-3.2-3B: index 0=embedding, 1-28=transformer blocks
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

In [ ]:
print(f"Device : {DEVICE} | {torch.cuda.get_device_name(0)}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    output_hidden_states=True,
)
model.eval()
print(f"Model loaded ✓  |  Total layers (incl. embedding): {NUM_LAYERS + 1}")

── CELL 3 — Core framework functions ───────────────────────────
These are the only functions you ever call.
No layer numbers appear outside this cell.

In [ ]:
# ── Text utilities ────────────────────────────────────────────────
def normalize(text):
    text = text.lower().strip()
    text = re.sub(r'\b(1\d{2}|\d{2})0s?\b',
                  lambda m: m.group(0).rstrip('s'), text)
    text = re.sub(r'\b(the|a|an)\b', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(text.split())

In [ ]:
def token_f1(pred, ref):
    p_tok = normalize(pred).split()
    r_tok = normalize(ref).split()
    if not p_tok or not r_tok: return 0.0
    common = set(p_tok) & set(r_tok)
    if not common: return 0.0
    p = len(common) / len(p_tok)
    r = len(common) / len(r_tok)
    return 2 * p * r / (p + r)

In [ ]:
def generate_response(question, max_new_tokens=40):
    prompt = (
        "Answer with just the name or short phrase, nothing else.\n"
        f"Question: {question}\nAnswer:"
    )
    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=200
    ).to(DEVICE)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.8,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return decoded.split('\n')[0].split('.')[0].strip()

In [ ]:
# ── Hidden state extraction ────────────────────────────────────────
def extract_hidden_states(data, layer_idx, cutoff_pct=1.0):
    """
    Extract hidden state at layer_idx, last token position.

    layer_idx  : 0=embedding, 1..NUM_LAYERS=transformer blocks
    cutoff_pct : 1.0=full sequence, 0.2=first 20% of tokens

    Returns:
        hiddens (N, hidden_dim) float32 cpu tensor
        labels  (N,)            int tensor  1=factual 0=hallucinated
    """
    hiddens, labels = [], []
    for item in data:
        full_text = item['prompt'] + " " + item['response']
        inputs    = tokenizer(
            full_text, return_tensors='pt',
            truncation=True, max_length=256
        ).to(DEVICE)

        if cutoff_pct < 1.0:
            total  = inputs['input_ids'].shape[1]
            cutoff = max(4, int(total * cutoff_pct))
            inputs = {k: v[:, :cutoff] for k, v in inputs.items()}

        with torch.no_grad():
            out = model(**inputs, output_hidden_states=True)

        h = out.hidden_states[layer_idx][0, -1, :].float().cpu()
        hiddens.append(h)
        labels.append(1 if item['factuality_score'] > 0.5 else 0)

    return torch.stack(hiddens), torch.tensor(labels)

In [ ]:
# ── AUROC utilities ───────────────────────────────────────────────
def quick_auroc(hiddens, labels, cv=3):
    """Fast 3-fold AUROC — used only for layer selection."""
    X = hiddens.numpy()
    y = labels.numpy()
    if len(np.unique(y)) < 2: return 0.5
    clf    = LogisticRegression(max_iter=300, C=1.0)
    scores = cross_val_score(clf, X, y, cv=cv, scoring='roc_auc')
    return float(scores.mean())

In [ ]:
def robust_auroc(hiddens, labels, cv=10):
    """Rigorous 10-fold stratified AUROC — used for final results."""
    X   = hiddens.numpy()
    y   = labels.numpy()
    if len(np.unique(y)) < 2: return 0.5, 0.0
    skf    = StratifiedKFold(
        n_splits=cv, shuffle=True, random_state=RANDOM_SEED
    )
    scores = []
    for tr, te in skf.split(X, y):
        clf = LogisticRegression(max_iter=1000, C=1.0)
        clf.fit(X[tr], y[tr])
        proba = clf.predict_proba(X[te])[:, 1]
        scores.append(roc_auc_score(y[te], proba))
    return float(np.mean(scores)), float(np.std(scores))

In [ ]:
# ── Core framework: automatic layer selection ──────────────────────
def find_and_evaluate(data, cutoff_pct=1.0, verbose=True):
    """
    Automatically finds L* and returns unbiased AUROC.
    No hardcoded layer numbers.

    Three-way split to prevent leakage:
      20% → layer selection  (sweep all layers, pick argmax)
      60% → probe training   (train logistic probe at L*)
      20% → evaluation       (final held-out AUROC)

    Parameters:
        data       : list of dicts — prompt, response, factuality_score
        cutoff_pct : fraction of tokens to use [0.1 .. 1.0]
        verbose    : print progress per layer

    Returns dict:
        best_layer   : int   automatically found L*
        layer_depth  : float L* / NUM_LAYERS  e.g. 0.82
        layer_aurocs : dict  {layer_idx: quick_auroc} for all layers
        probe_auroc  : float final unbiased AUROC on held-out set
        probe_std    : float std of probe AUROC
        probe        : trained LogisticRegression
        h_eval       : hidden states on eval set (for PCA etc.)
        l_eval       : labels on eval set
    """
    n = len(data)
    sel_end   = int(0.2 * n)
    train_end = int(0.8 * n)

    sel_data   = data[:sel_end]           # 20%
    train_data = data[sel_end:train_end]  # 60%
    eval_data  = data[train_end:]         # 20%

    if verbose:
        print(f"  Split → sel:{len(sel_data)} "
              f"| train:{len(train_data)} "
              f"| eval:{len(eval_data)}")

    # ── step 1: sweep all layers on selection set ────────────────
    if verbose:
        print(f"  Sweeping {NUM_LAYERS+1} layers "
              f"at cutoff={int(cutoff_pct*100)}% ...")

    layer_aurocs = {}
    for layer_idx in range(NUM_LAYERS + 1):
        h, l = extract_hidden_states(sel_data, layer_idx, cutoff_pct)
        layer_aurocs[layer_idx] = quick_auroc(h, l, cv=3)

    best_layer = max(layer_aurocs, key=layer_aurocs.get)
    if verbose:
        print(f"  L* = layer {best_layer} "
              f"({best_layer/NUM_LAYERS*100:.0f}% depth) | "
              f"sel AUROC = {layer_aurocs[best_layer]:.4f}")

    # ── step 2: train probe at L* on train set ───────────────────
    h_train, l_train = extract_hidden_states(
        train_data, best_layer, cutoff_pct
    )
    probe = LogisticRegression(max_iter=1000, C=1.0)
    probe.fit(h_train.numpy(), l_train.numpy())

    # ── step 3: evaluate on held-out eval set ────────────────────
    h_eval, l_eval = extract_hidden_states(
        eval_data, best_layer, cutoff_pct
    )
    proba      = probe.predict_proba(h_eval.numpy())[:, 1]
    probe_auroc = roc_auc_score(l_eval.numpy(), proba)
    _, probe_std = robust_auroc(h_eval, l_eval, cv=5)

    if verbose:
        print(f"  Final AUROC = {probe_auroc:.4f} ± {probe_std:.4f}")

    return {
        'best_layer':   best_layer,
        'layer_depth':  best_layer / NUM_LAYERS,
        'layer_aurocs': layer_aurocs,
        'probe_auroc':  probe_auroc,
        'probe_std':    probe_std,
        'probe':        probe,
        'h_eval':       h_eval,
        'l_eval':       l_eval,
    }

In [ ]:
print("Core framework functions loaded ✓")

── CELL 4 — Dataset loaders ─────────────────────────────────────

In [ ]:
def load_triviaqa(n=400):
    """Reuse cached labeled_data.json from previous experiment."""
    path = f'{SAVE_DIR}/labeled_data.json'
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        random.seed(RANDOM_SEED)
        random.shuffle(data)
        print(f"TriviaQA  : {len(data[:n])} samples (from cache)")
        return data[:n]
    raise FileNotFoundError(
        "labeled_data.json not found. "
        "Run the data generation cell from facr_colab.py first."
    )

In [ ]:
def load_halueval(n=400):
    """
    HaluEval QA: pre-labeled factual + hallucinated pairs.
    50/50 balanced by construction — no generation needed.
    """
    path = f'{STEP1_DIR}/halueval_data.json'
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        print(f"HaluEval  : {len(data[:n])} samples (from cache)")
        return data[:n]

    ds   = load_dataset("pminervini/HaluEval", "qa_samples", split="data")
    data = []
    for ex in list(ds):
        question  = ex.get('question', '').strip()
        right_ans = ex.get('right_answer', '').strip()
        hallu_ans = ex.get('hallucinated_answer', '').strip()
        if not question or not right_ans or not hallu_ans:
            continue
        prompt = (
            "Answer with just the name or short phrase, nothing else.\n"
            f"Question: {question}\nAnswer:"
        )
        data.append({
            'question': question, 'response': right_ans,
            'factuality_score': 1.0, 'prompt': prompt
        })
        data.append({
            'question': question, 'response': hallu_ans,
            'factuality_score': 0.0, 'prompt': prompt
        })
        if len(data) >= n: break

    random.seed(RANDOM_SEED)
    random.shuffle(data)
    with open(path, 'w') as f: json.dump(data[:n], f)
    print(f"HaluEval  : {len(data[:n])} samples (50/50 by construction)")
    return data[:n]

In [ ]:
def load_truthfulqa(n=400):
    """
    TruthfulQA: adversarially constructed questions.
    best_answer=factual, incorrect_answers[0]=hallucinated.
    """
    path = f'{STEP1_DIR}/truthfulqa_data.json'
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        print(f"TruthfulQA: {len(data[:n])} samples (from cache)")
        return data[:n]

    ds   = load_dataset("truthful_qa", "generation", split="validation")
    data = []
    for ex in ds:
        question   = ex.get('question', '').strip()
        best_ans   = ex.get('best_answer', '').strip()
        wrong_list = ex.get('incorrect_answers', [])
        if not question or not best_ans or not wrong_list: continue
        prompt = (
            "Answer with just the name or short phrase, nothing else.\n"
            f"Question: {question}\nAnswer:"
        )
        data.append({
            'question': question, 'response': best_ans,
            'factuality_score': 1.0, 'prompt': prompt
        })
        data.append({
            'question': question, 'response': wrong_list[0],
            'factuality_score': 0.0, 'prompt': prompt
        })
        if len(data) >= n: break

    random.seed(RANDOM_SEED)
    random.shuffle(data)
    with open(path, 'w') as f: json.dump(data[:n], f)
    print(f"TruthfulQA: {len(data[:n])} samples")
    return data[:n]

In [ ]:
def load_nq_open(n=400):
    """
    NQ-Open: real Google search queries.
    Generates responses and labels with token F1 against gold answers.
    """
    path = f'{STEP1_DIR}/nq_labeled.json'
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        print(f"NQ-Open   : {len(data[:n])} samples (from cache)")
        return data[:n]

    ds   = load_dataset("nq_open", split="train[:300]")
    data = []
    print("NQ-Open: generating responses (cached after this run)...")
    for ex in tqdm(ds):
        question  = ex['question']
        gold_list = ex['answer']
        prompt    = (
            "Answer with just the name or short phrase, nothing else.\n"
            f"Question: {question}\nAnswer:"
        )
        for _ in range(2):
            resp  = generate_response(question)
            score = max(token_f1(resp, ref) for ref in gold_list)
            data.append({
                'question': question, 'response': resp,
                'factuality_score': float(score), 'prompt': prompt
            })
        if len(data) >= n: break

    random.seed(RANDOM_SEED)
    random.shuffle(data)
    with open(path, 'w') as f: json.dump(data[:n], f)
    print(f"NQ-Open   : {len(data[:n])} samples generated and cached")
    return data[:n]

In [ ]:
# ── Load all datasets ─────────────────────────────────────────────
print("Loading datasets...")
datasets = {
    'TriviaQA':   load_triviaqa(n=400),
    'HaluEval':   load_halueval(n=400),
    'TruthfulQA': load_truthfulqa(n=400),
    'NQ-Open':    load_nq_open(n=400),
}
print("\nDataset summary:")
for name, data in datasets.items():
    f = sum(1 for d in data if d['factuality_score'] > 0.5)
    h = len(data) - f
    print(f"  {name:<12}: {len(data):>4} samples | "
          f"factual={f} ({100*f/len(data):.0f}%) | "
          f"hallu={h} ({100*h/len(data):.0f}%)")

── CELL 5 — EXPERIMENT A: full sequence, all datasets ───────────
Runs find_and_evaluate on every dataset at 100% token cutoff.
Answers: does L* stay at ~80% depth across datasets?
Expected runtime: ~20 min

In [ ]:
print("\n" + "="*60)
print("  EXPERIMENT A — Full sequence, all 4 datasets")
print("  Automatic L* selection per dataset")
print("="*60)

In [ ]:
full_results = {}
for ds_name, ds_data in datasets.items():
    print(f"\n── {ds_name} " + "─"*40)
    result = find_and_evaluate(ds_data, cutoff_pct=1.0, verbose=True)
    full_results[ds_name] = result

In [ ]:
# save (exclude non-serializable objects)
save_exp_a = {
    k: {
        'best_layer':   v['best_layer'],
        'layer_depth':  v['layer_depth'],
        'layer_aurocs': {str(i): a for i, a in v['layer_aurocs'].items()},
        'probe_auroc':  v['probe_auroc'],
        'probe_std':    v['probe_std'],
    }
    for k, v in full_results.items()
}
with open(f'{STEP1_DIR}/exp_a_results.json', 'w') as f:
    json.dump(save_exp_a, f, indent=2)

In [ ]:
print("\n── Experiment A Summary ──────────────────────────────────")
print(f"{'Dataset':<14} {'L*':>4} {'Depth':>8} {'AUROC':>8} {'±Std':>8}")
print("-" * 48)
for ds_name, result in full_results.items():
    print(f"{ds_name:<14} "
          f"{result['best_layer']:>4} "
          f"{result['layer_depth']*100:>7.0f}% "
          f"{result['probe_auroc']:>8.4f} "
          f"±{result['probe_std']:.4f}")

── CELL 6 — EXPERIMENT B: early detection sweep ─────────────────
Runs find_and_evaluate at 5 token cutoffs on TriviaQA + HaluEval.
L* is re-selected at EACH cutoff independently.
Answers: how early can we reliably detect? Does L* shift?
Expected runtime: ~25 min

In [ ]:
CUTOFFS = [0.10, 0.20, 0.30, 0.50, 1.00]

In [ ]:
print("\n" + "="*60)
print("  EXPERIMENT B — Early detection sweep")
print("  L* re-selected at each cutoff independently")
print("="*60)

In [ ]:
early_results = {}
for ds_name in ['TriviaQA', 'HaluEval']:
    early_results[ds_name] = {}
    print(f"\n── {ds_name} " + "─"*40)
    for cutoff in CUTOFFS:
        print(f"\n  Cutoff = {int(cutoff*100)}% tokens:")
        result = find_and_evaluate(
            datasets[ds_name], cutoff_pct=cutoff, verbose=True
        )
        early_results[ds_name][cutoff] = {
            'best_layer':  result['best_layer'],
            'layer_depth': result['layer_depth'],
            'probe_auroc': result['probe_auroc'],
            'probe_std':   result['probe_std'],
        }

In [ ]:
with open(f'{STEP1_DIR}/exp_b_results.json', 'w') as f:
    json.dump(early_results, f, indent=2)

In [ ]:
print("\n── Experiment B Summary ──────────────────────────────────")
for ds_name in ['TriviaQA', 'HaluEval']:
    print(f"\n  {ds_name}:")
    print(f"  {'Cutoff':>8} {'L*':>4} {'Depth':>8} {'AUROC':>8} {'±Std':>8}")
    print("  " + "-"*44)
    for cutoff in CUTOFFS:
        r = early_results[ds_name][cutoff]
        print(f"  {int(cutoff*100):>6}%  "
              f"{r['best_layer']:>4}  "
              f"{r['layer_depth']*100:>7.0f}%  "
              f"{r['probe_auroc']:>8.4f}  "
              f"±{r['probe_std']:.4f}")

── CELL 7 — EXPERIMENT C: cross-dataset transfer ────────────────
Train probe on TriviaQA's L* → test on all other datasets.
Answers: does the geometry transfer across datasets?
Expected runtime: ~5 min

In [ ]:
print("\n" + "="*60)
print("  EXPERIMENT C — Cross-dataset transfer")
print("  Probe trained on TriviaQA, tested on all datasets")
print("="*60)

In [ ]:
# get TriviaQA's automatically found L*
tqa_best_layer = full_results['TriviaQA']['best_layer']
print(f"\nTriviaQA L* = {tqa_best_layer} "
      f"({tqa_best_layer/NUM_LAYERS*100:.0f}% depth)")

In [ ]:
# train probe on ALL TriviaQA data at that layer
print("Training transfer probe on full TriviaQA dataset...")
h_all_tqa, l_all_tqa = extract_hidden_states(
    datasets['TriviaQA'], tqa_best_layer, cutoff_pct=1.0
)
transfer_probe = LogisticRegression(max_iter=1000, C=1.0)
transfer_probe.fit(h_all_tqa.numpy(), l_all_tqa.numpy())
print("Transfer probe trained ✓")

In [ ]:
# test on all datasets at TriviaQA's L*
transfer_results = {}
print(f"\n{'Dataset':<14} {'Transfer':>10} {'Own L*':>8} {'Own AUROC':>10} {'Gap':>8}")
print("-" * 55)
for ds_name, ds_data in datasets.items():
    h, l  = extract_hidden_states(ds_data, tqa_best_layer, 1.0)
    proba = transfer_probe.predict_proba(h.numpy())[:, 1]
    t_auroc = roc_auc_score(l.numpy(), proba)

    own_layer = full_results[ds_name]['best_layer']
    own_auroc = full_results[ds_name]['probe_auroc']
    gap       = own_auroc - t_auroc

    transfer_results[ds_name] = {
        'transfer_auroc': t_auroc,
        'own_auroc':      own_auroc,
        'own_layer':      own_layer,
        'transfer_layer': tqa_best_layer,
        'gap':            gap,
    }
    note = "← train" if ds_name == 'TriviaQA' else "transfer"
    print(f"{ds_name:<14} {t_auroc:>10.4f} "
          f"{own_layer:>8} {own_auroc:>10.4f} "
          f"{gap:>+8.4f}  {note}")

In [ ]:
with open(f'{STEP1_DIR}/exp_c_results.json', 'w') as f:
    json.dump(transfer_results, f, indent=2)

── CELL 8 — Figures ─────────────────────────────────────────────

In [ ]:
COLORS = {
    'TriviaQA':   'steelblue',
    'HaluEval':   'coral',
    'TruthfulQA': 'mediumseagreen',
    'NQ-Open':    'mediumpurple',
}

In [ ]:
# ── Figure 1: Layer AUROC curves per dataset ──────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=True)
for ax, (ds_name, result) in zip(axes, full_results.items()):
    color  = COLORS[ds_name]
    layers = sorted(result['layer_aurocs'].keys())
    aurocs = [result['layer_aurocs'][l] for l in layers]
    best   = result['best_layer']

    ax.plot(layers, aurocs, '-', color=color, linewidth=1.5, alpha=0.8)
    ax.scatter(layers, aurocs, color=color, s=20, zorder=4)
    ax.axvline(best, color='black', linestyle='--',
               linewidth=1.2, alpha=0.7)
    ax.scatter([best], [result['layer_aurocs'][best]],
               color='black', s=100, zorder=5,
               label=f"L*={best} ({best/NUM_LAYERS*100:.0f}%)")
    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
    ax.set_title(f"{ds_name}\nAUROC={result['probe_auroc']:.3f}", fontsize=10)
    ax.set_xlabel('Layer', fontsize=9)
    ax.set_ylim(0.4, 1.05)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.2)
    if ax == axes[0]:
        ax.set_ylabel('AUROC (probe)', fontsize=10)

In [ ]:
plt.suptitle(
    'Automatic layer selection (L*) across 4 datasets\n'
    'Dashed = L* found by framework, no hardcoded layer numbers',
    fontsize=12
)
plt.tight_layout()
plt.savefig(f'{STEP1_DIR}/fig1_layer_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 1 saved ✓")

In [ ]:
# ── Figure 2: Where does L* fall? ────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ds_names = list(full_results.keys())
depths   = [full_results[n]['layer_depth'] * 100 for n in ds_names]
aurocs_f = [full_results[n]['probe_auroc']       for n in ds_names]
colors_b = [COLORS[n] for n in ds_names]
x        = np.arange(len(ds_names))

In [ ]:
bars = ax.bar(x, depths, color=colors_b, edgecolor='white', width=0.5)
for bar, auroc, depth, ds_name in zip(bars, aurocs_f, depths, ds_names):
    layer = full_results[ds_name]['best_layer']
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 1.5,
            f'L*={layer}\nAUROC={auroc:.3f}',
            ha='center', va='bottom', fontsize=9)

In [ ]:
ax.axhspan(75, 85, alpha=0.08, color='green', label='75-85% band')
ax.axhline(80, color='green', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(ds_names, fontsize=10)
ax.set_ylabel('L* depth (% of network)', fontsize=11)
ax.set_ylim(0, 115)
ax.set_title(
    'Where does L* fall across datasets?\n'
    'Green band = 75-85% (predicted universal range)',
    fontsize=11
)
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{STEP1_DIR}/fig2_layer_depth.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 2 saved ✓")

In [ ]:
# ── Figure 3: Early detection curves ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, ds_name in zip(axes, ['TriviaQA', 'HaluEval']):
    cutoffs    = sorted(early_results[ds_name].keys())
    auroc_vals = [early_results[ds_name][c]['probe_auroc'] for c in cutoffs]
    std_vals   = [early_results[ds_name][c]['probe_std']   for c in cutoffs]
    layers     = [early_results[ds_name][c]['best_layer']  for c in cutoffs]
    x_labels   = [f"{int(c*100)}%" for c in cutoffs]
    xi         = range(len(cutoffs))

    ax.plot(xi, auroc_vals, 'o-', color=COLORS[ds_name],
            linewidth=2, markersize=8)
    ax.fill_between(xi,
        [a-s for a, s in zip(auroc_vals, std_vals)],
        [a+s for a, s in zip(auroc_vals, std_vals)],
        alpha=0.15, color=COLORS[ds_name])

    for i, (a, l) in enumerate(zip(auroc_vals, layers)):
        ax.annotate(f"L*={l}", (i, a),
                    textcoords="offset points",
                    xytext=(0, 10), ha='center',
                    fontsize=8, color=COLORS[ds_name])

    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Chance')
    ax.axhline(0.7, color='green', linestyle='--', alpha=0.5, label='0.7 threshold')
    ax.set_xticks(xi)
    ax.set_xticklabels(x_labels, fontsize=10)
    ax.set_xlabel('Tokens generated (% of full response)', fontsize=10)
    ax.set_ylabel('AUROC', fontsize=10)
    ax.set_title(f'{ds_name} — early detection\n'
                 f'(L* re-selected at each cutoff)',
                 fontsize=10)
    ax.set_ylim(0.4, 1.05)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

In [ ]:
plt.suptitle(
    'How early can hallucination be detected?\n'
    'L* label = automatically selected layer at each cutoff',
    fontsize=12
)
plt.tight_layout()
plt.savefig(f'{STEP1_DIR}/fig3_early_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 3 saved ✓")

In [ ]:
# ── Figure 4: Transfer bar chart ──────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ds_list   = list(transfer_results.keys())
own_vals  = [transfer_results[d]['own_auroc']      for d in ds_list]
xfer_vals = [transfer_results[d]['transfer_auroc'] for d in ds_list]
x = np.arange(len(ds_list))
w = 0.35

In [ ]:
ax.bar(x - w/2, own_vals,  w, label='Own L*',
       color='steelblue', alpha=0.85)
ax.bar(x + w/2, xfer_vals, w, label=f'TriviaQA L*={tqa_best_layer} (transfer)',
       color='coral',     alpha=0.85)
for i, (o, t) in enumerate(zip(own_vals, xfer_vals)):
    ax.text(i-w/2, o+0.005, f'{o:.3f}', ha='center', va='bottom', fontsize=8)
    ax.text(i+w/2, t+0.005, f'{t:.3f}', ha='center', va='bottom', fontsize=8)

In [ ]:
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(ds_list, fontsize=9)
ax.set_ylabel('AUROC', fontsize=11)
ax.set_ylim(0.4, 1.1)
ax.set_title(
    'Own L* vs transferred L* from TriviaQA\n'
    'Gap = cost of using a fixed layer across datasets',
    fontsize=10
)
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{STEP1_DIR}/fig4_transfer.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 4 saved ✓")

── CELL 9 — INTROSPECTION ────────────────────────────────────────
Reads all results and draws conclusions.
Tells you what to write in the paper and whether to adjust the plan.

In [ ]:
print("\n" + "="*60)
print("  INTROSPECTION — What did we find?")
print("="*60)

In [ ]:
# finding 1: is L* consistent?
depths_pct = {n: full_results[n]['layer_depth']*100 for n in full_results}
mean_depth = np.mean(list(depths_pct.values()))
std_depth  = np.std(list(depths_pct.values()))

In [ ]:
print(f"\n1. Does L* stay at consistent depth across datasets?")
for ds_name, depth in depths_pct.items():
    layer = full_results[ds_name]['best_layer']
    print(f"   {ds_name:<14}: L*={layer:2d}  ({depth:.0f}% depth)")
print(f"\n   Mean depth : {mean_depth:.1f}% ± {std_depth:.1f}%")

In [ ]:
if std_depth < 8:
    finding_1 = "universal"
    print("   ✓ CONSISTENT — L* clusters at same depth across datasets")
    print("     → This is an architectural property of the model")
elif std_depth < 15:
    finding_1 = "mostly-consistent"
    print("   ~ MOSTLY CONSISTENT — some variation but clear trend")
else:
    finding_1 = "task-dependent"
    print("   ~ TASK-DEPENDENT — L* shifts significantly across datasets")
    print("     → The shift itself is the finding")

In [ ]:
# finding 2: probe transfer
xfer_aurocs = {k: v['transfer_auroc']
               for k, v in transfer_results.items()
               if k != 'TriviaQA'}
mean_transfer = np.mean(list(xfer_aurocs.values()))

In [ ]:
print(f"\n2. Does the probe transfer across datasets?")
for ds_name, auroc in xfer_aurocs.items():
    own = transfer_results[ds_name]['own_auroc']
    gap = own - auroc
    status = "✓" if auroc > 0.65 else ("~" if auroc > 0.55 else "✗")
    print(f"   {ds_name:<14}: transfer={auroc:.4f} "
          f"own={own:.4f} gap={gap:+.4f}  {status}")
print(f"\n   Mean transfer AUROC: {mean_transfer:.4f}")

In [ ]:
if mean_transfer > 0.65:
    finding_2 = "generalizes"
    print("   ✓ GENERALIZES — probe transfers well")
elif mean_transfer > 0.55:
    finding_2 = "partial"
    print("   ~ PARTIAL — some transfer, own L* still better")
else:
    finding_2 = "no-transfer"
    print("   ✗ DOES NOT TRANSFER — geometry is dataset-specific")

In [ ]:
# finding 3: earliest reliable cutoff
print(f"\n3. Earliest reliable detection cutoff?")
for ds_name in ['TriviaQA', 'HaluEval']:
    print(f"\n   {ds_name}:")
    for cutoff in sorted(early_results[ds_name].keys()):
        r = early_results[ds_name][cutoff]
        a = r['probe_auroc']
        l = r['best_layer']
        d = r['layer_depth'] * 100
        reliable = "✓ reliable" if a > 0.65 else "✗ not yet"
        print(f"   {int(cutoff*100):>4}% tokens | "
              f"L*={l} ({d:.0f}%) | AUROC={a:.4f}  {reliable}")

In [ ]:
# decision
print(f"\n4. Decision for next step:")
if finding_1 in ("universal", "mostly-consistent") and finding_2 in ("generalizes", "partial"):
    print("   → PROCEED to Experiment D: contrastive loss")
    print("   → Apply InfoNCE at auto-selected L* (no hardcoding)")
    print("   → Expect: geometry sharpens at L*, AUROC improves")
elif finding_1 == "task-dependent":
    print("   → INTERESTING: L* shifts are the contribution")
    print("   → Proceed to Experiment D but report per-dataset L*")
else:
    print("   → Diagnose why transfer fails before proceeding")

In [ ]:
print("\n" + "="*60)

── CELL 10 — EXPERIMENT D: contrastive loss (decoupled) ─────────
Train ONLY the contrastive objective at the auto-selected L*.
Model weights are FROZEN — only a projection head is trained.
This avoids the SFT+contrastive collapse we saw earlier.
Expected runtime: ~15 min

In [ ]:
print("\n" + "="*60)
print("  EXPERIMENT D — Contrastive geometry sharpening")
print("  Frozen model + trainable projection head at L*")
print("="*60)

In [ ]:
# ── Projection head ───────────────────────────────────────────────
class ProjectionHead(torch.nn.Module):
    """
    Lightweight MLP that projects hidden states into a
    geometry-shaped space where factual/hallucinated are separated.
    Only this head is trained — base model stays frozen.
    """
    def __init__(self, input_dim, proj_dim=256):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, proj_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(proj_dim, proj_dim)
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)   # unit sphere

In [ ]:
# ── InfoNCE contrastive loss ──────────────────────────────────────
def infonce_loss(z, factuality_scores, temperature=0.07):
    """
    InfoNCE-style soft contrastive loss.
    Always positive — prevents collapse.

    z                : (batch, proj_dim) — normalized projections
    factuality_scores: (batch,)          — float [0,1]
    temperature      : scale factor      — smaller = sharper

    Loss is low when similar-factuality samples are close
    and different-factuality samples are far apart.
    """
    sim = torch.matmul(z, z.T) / temperature   # (batch, batch)

    # soft positive weights from factuality proximity
    f       = factuality_scores.unsqueeze(0) - factuality_scores.unsqueeze(1)
    weights = torch.exp(-f**2 / (2 * 0.5**2))
    weights = weights / weights.sum(dim=1, keepdim=True).clamp(min=1e-8)

    # InfoNCE cross-entropy: minimize KL(weights || softmax(sim))
    log_probs = F.log_softmax(sim, dim=1)
    loss      = -(weights * log_probs).sum(dim=1).mean()
    return loss

In [ ]:
# ── Training dataset ──────────────────────────────────────────────
class ContrastiveDataset(TorchDataset):
    def __init__(self, hiddens, factuality_scores):
        self.hiddens  = hiddens
        self.scores   = factuality_scores

    def __len__(self):
        return len(self.hiddens)

    def __getitem__(self, idx):
        return {
            'hidden':            self.hiddens[idx],
            'factuality_score':  self.scores[idx]
        }

In [ ]:
def train_projection_head(train_data, target_layer,
                           proj_dim=256, epochs=10,
                           batch_size=64, lr=1e-3,
                           temperature=0.07):
    """
    Train only the projection head at target_layer.
    Base model stays completely frozen.
    Returns trained projection head.
    """
    # extract hidden states for all training data
    print(f"  Extracting hidden states at layer {target_layer}...")
    h_train, l_train = extract_hidden_states(
        train_data, target_layer, cutoff_pct=1.0
    )
    hidden_dim = h_train.shape[1]

    # balance classes for better contrastive signal
    f_idx = (l_train == 1).nonzero().squeeze()
    h_idx = (l_train == 0).nonzero().squeeze()
    min_n = min(len(f_idx), len(h_idx))
    bal_idx = torch.cat([f_idx[:min_n], h_idx[:min_n]])
    bal_idx = bal_idx[torch.randperm(len(bal_idx))]

    h_bal = h_train[bal_idx]
    s_bal = l_train[bal_idx].float()

    dataset    = ContrastiveDataset(h_bal, s_bal)
    dataloader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=True
    )

    # projection head
    head      = ProjectionHead(hidden_dim, proj_dim).to(DEVICE)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs
    )

    print(f"  Training projection head "
          f"({sum(p.numel() for p in head.parameters()):,} params) "
          f"for {epochs} epochs...")

    head.train()
    for epoch in range(epochs):
        epoch_loss = 0
        for batch in dataloader:
            h = batch['hidden'].to(DEVICE)
            s = batch['factuality_score'].to(DEVICE)

            optimizer.zero_grad()
            z    = head(h)
            loss = infonce_loss(z, s, temperature)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(dataloader)
        if (epoch + 1) % 2 == 0:
            print(f"  Epoch {epoch+1:2d}/{epochs} | loss={avg_loss:.4f}")

        scheduler.step()

    head.eval()
    print("  Projection head trained ✓")
    return head, hidden_dim

In [ ]:
# ── Run Experiment D on TriviaQA ─────────────────────────────────
# Use TriviaQA's automatically found L*
tqa_layer  = full_results['TriviaQA']['best_layer']
tqa_train  = datasets['TriviaQA'][:int(0.8 * len(datasets['TriviaQA']))]
tqa_eval   = datasets['TriviaQA'][int(0.8 * len(datasets['TriviaQA'])):]

In [ ]:
print(f"\nTarget layer: {tqa_layer} "
      f"({tqa_layer/NUM_LAYERS*100:.0f}% depth) "
      f"[auto-selected from TriviaQA]")

In [ ]:
proj_head, hidden_dim = train_projection_head(
    tqa_train, tqa_layer,
    proj_dim=256, epochs=10,
    batch_size=64, lr=1e-3
)

In [ ]:
# Save projection head
torch.save(proj_head.state_dict(),
           f'{STEP2_DIR}/proj_head_layer{tqa_layer}.pt')
print(f"Projection head saved ✓")

── CELL 11 — EXPERIMENT E: before vs after geometry ─────────────
Compare probe AUROC before (raw hidden states) vs
after (projected through trained head).
Also visualize PCA of raw vs projected space.

In [ ]:
print("\n" + "="*60)
print("  EXPERIMENT E — Before vs after contrastive geometry")
print("="*60)

In [ ]:
def evaluate_with_head(head, data, target_layer, cutoff_pct=1.0):
    """
    Extract hidden states at target_layer,
    project through head, then compute probe AUROC.
    """
    h, l = extract_hidden_states(data, target_layer, cutoff_pct)
    # project
    with torch.no_grad():
        z = head(h.to(DEVICE)).cpu()

    mean, std = robust_auroc(z, l, cv=10)
    return z, l, mean, std

In [ ]:
contrastive_results = {}

In [ ]:
print("\nEvaluating on all datasets + cutoffs...")
for ds_name, ds_data in datasets.items():
    print(f"\n── {ds_name} ──────────────────────────────────────────")
    ds_eval = ds_data[int(0.8*len(ds_data)):]   # held-out 20%

    # before: raw hidden states at tqa_layer
    h_raw, l_raw   = extract_hidden_states(ds_eval, tqa_layer, 1.0)
    raw_mean, raw_std = robust_auroc(h_raw, l_raw, cv=10)

    # after: projected hidden states
    z_proj, l_proj, proj_mean, proj_std = evaluate_with_head(
        proj_head, ds_eval, tqa_layer, 1.0
    )

    delta = proj_mean - raw_mean
    contrastive_results[ds_name] = {
        'raw_auroc':  raw_mean,
        'raw_std':    raw_std,
        'proj_auroc': proj_mean,
        'proj_std':   proj_std,
        'delta':      delta,
        'h_raw':      h_raw,
        'z_proj':     z_proj,
        'labels':     l_raw,
    }
    print(f"  Raw hidden    : AUROC = {raw_mean:.4f} ± {raw_std:.4f}")
    print(f"  Projected     : AUROC = {proj_mean:.4f} ± {proj_std:.4f}")
    print(f"  Delta         : {delta:+.4f}")

In [ ]:
# early detection with projection head
print("\n── Early detection: raw vs projected (TriviaQA) ─────────")
print(f"{'Cutoff':>8} {'Raw AUROC':>12} {'Proj AUROC':>12} {'Delta':>8}")
print("-" * 46)

In [ ]:
early_contrastive = {}
for cutoff in CUTOFFS:
    ds_eval = datasets['TriviaQA'][int(0.8*len(datasets['TriviaQA'])):]

    h_raw, l_raw      = extract_hidden_states(ds_eval, tqa_layer, cutoff)
    raw_mean, _       = robust_auroc(h_raw, l_raw, cv=5)

    z_proj, l_proj, proj_mean, _ = evaluate_with_head(
        proj_head, ds_eval, tqa_layer, cutoff
    )

    delta = proj_mean - raw_mean
    early_contrastive[cutoff] = {
        'raw_auroc':  raw_mean,
        'proj_auroc': proj_mean,
        'delta':      delta,
    }
    print(f"{int(cutoff*100):>7}%  {raw_mean:>12.4f} "
          f"{proj_mean:>12.4f} {delta:>+8.4f}")

In [ ]:
res = contrastive_results['TriviaQA']
for ax, hiddens, title in [
    (axes[0], res['h_raw'],  f'Raw hidden states\n(layer {tqa_layer})'),
    (axes[1], res['z_proj'], 'After contrastive projection')
]:
    y       = res['labels'].numpy()
    pca     = PCA(n_components=2)
    reduced = pca.fit_transform(hiddens.numpy())

    f_mask = y == 1
    ax.scatter(reduced[f_mask,  0], reduced[f_mask,  1],
               c='steelblue', alpha=0.6, s=18, label='Factual')
    ax.scatter(reduced[~f_mask, 0], reduced[~f_mask, 1],
               c='coral',     alpha=0.6, s=18, label='Hallucinated')

    # centroid similarity
    h_t = torch.tensor(hiddens.numpy() if not isinstance(hiddens, np.ndarray)
                       else hiddens)
    fc  = h_t[f_mask].mean(0)
    hc  = h_t[~f_mask].mean(0)
    sim = F.cosine_similarity(fc.unsqueeze(0), hc.unsqueeze(0)).item()

    _, auroc_std = robust_auroc(
        torch.tensor(hiddens.numpy()
                     if not isinstance(hiddens, np.ndarray) else hiddens),
        res['labels'], cv=5
    )
    ax.set_title(f'{title}\nCentroid sim={sim:.3f}', fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

In [ ]:
plt.suptitle(
    f'Activation geometry before vs after contrastive projection\n'
    f'Layer {tqa_layer} ({tqa_layer/NUM_LAYERS*100:.0f}% depth) — TriviaQA',
    fontsize=12
)
plt.tight_layout()
plt.savefig(f'{STEP2_DIR}/fig5_pca_before_after.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure 5 saved ✓")

In [ ]:
# ── Figure 6: Early detection before vs after ─────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

In [ ]:
cutoffs    = sorted(early_contrastive.keys())
raw_vals   = [early_contrastive[c]['raw_auroc']  for c in cutoffs]
proj_vals  = [early_contrastive[c]['proj_auroc'] for c in cutoffs]
x_labels   = [f"{int(c*100)}%" for c in cutoffs]
xi         = range(len(cutoffs))

In [ ]:
ax.plot(xi, raw_vals,  'o-', color='steelblue', linewidth=2,
        markersize=8, label='Raw hidden states')
ax.plot(xi, proj_vals, 's-', color='coral',     linewidth=2,
        markersize=8, label='After contrastive projection')
ax.fill_between(xi,
    [max(0, p-r) for p, r in zip(proj_vals, raw_vals)],
    alpha=0.1, color='coral', label='Improvement')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.axhline(0.7, color='green', linestyle='--', alpha=0.5,
           label='0.7 threshold')
ax.set_xticks(xi)
ax.set_xticklabels(x_labels, fontsize=11)
ax.set_xlabel('Tokens generated (% of full response)', fontsize=11)
ax.set_ylabel('AUROC', fontsize=11)
ax.set_title(
    f'Early hallucination detection — before vs after\n'
    f'contrastive projection at layer {tqa_layer}',
    fontsize=11
)
ax.set_ylim(0.4, 1.05)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{STEP2_DIR}/fig6_early_detection_contrast.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure 6 saved ✓")

── CELL 12 — Final results table ────────────────────────────────
Paper-ready summary of all experiments.

In [ ]:
print("\n" + "="*60)
print("  FINAL RESULTS TABLE — PAPER READY")
print("="*60)

In [ ]:
print(f"\nModel  : Llama-3.2-3B-Instruct")
print(f"Method : Automatic L* selection + InfoNCE projection head")

In [ ]:
print(f"\n── Table 1: Layer selection results ─────────────────────")
print(f"{'Dataset':<14} {'L*':>4} {'Depth':>8} "
      f"{'AUROC':>8} {'±Std':>8}")
print("-" * 50)
for ds_name in datasets.keys():
    r = full_results[ds_name]
    print(f"{ds_name:<14} {r['best_layer']:>4} "
          f"{r['layer_depth']*100:>7.0f}%  "
          f"{r['probe_auroc']:>8.4f}  "
          f"±{r['probe_std']:.4f}")

In [ ]:
depths_list = [full_results[n]['layer_depth']*100 for n in datasets]
print(f"\n  L* depth: {np.mean(depths_list):.1f}% ± {np.std(depths_list):.1f}%")

In [ ]:
print(f"\n── Table 2: Early detection (TriviaQA, layer {tqa_layer}) ─")
print(f"{'Cutoff':>8} {'Raw AUROC':>12} {'Proj AUROC':>12} {'Delta':>8}")
print("-" * 46)
for cutoff in CUTOFFS:
    r = early_contrastive[cutoff]
    print(f"{int(cutoff*100):>7}%  "
          f"{r['raw_auroc']:>12.4f}  "
          f"{r['proj_auroc']:>12.4f}  "
          f"{r['delta']:>+8.4f}")

In [ ]:
print(f"\n── Table 3: Cross-dataset generalization ─────────────────")
print(f"{'Dataset':<14} {'Own L*':>6} {'Own AUROC':>10} "
      f"{'Transfer':>10} {'Gap':>8}")
print("-" * 52)
for ds_name in datasets.keys():
    t = transfer_results[ds_name]
    print(f"{ds_name:<14} {t['own_layer']:>6}  "
          f"{t['own_auroc']:>10.4f}  "
          f"{t['transfer_auroc']:>10.4f}  "
          f"{t['gap']:>+8.4f}")

In [ ]:
print(f"\n── Table 4: Contrastive projection improvement ───────────")
print(f"{'Dataset':<14} {'Raw':>8} {'Projected':>10} {'Delta':>8}")
print("-" * 44)
for ds_name, r in contrastive_results.items():
    print(f"{ds_name:<14} {r['raw_auroc']:>8.4f}  "
          f"{r['proj_auroc']:>10.4f}  "
          f"{r['delta']:>+8.4f}")

In [ ]:
print(f"\n── Key findings ──────────────────────────────────────────")
all_layers = [full_results[n]['best_layer'] for n in datasets]
print(f"1. L* identified automatically — no hardcoded layer numbers")
print(f"   Mean L* depth: {np.mean(depths_list):.1f}% ± {np.std(depths_list):.1f}%")
print(f"   Layers found : {all_layers}")
print(f"2. Cross-dataset transfer: {mean_transfer:.4f} mean AUROC")
print(f"3. Earliest reliable detection: see Table 2")
contrastive_deltas = [r['delta'] for r in contrastive_results.values()]
print(f"4. Contrastive projection: mean delta = "
      f"{np.mean(contrastive_deltas):+.4f}")

In [ ]:
print(f"\nAll figures saved to:")
print(f"  {STEP1_DIR}/fig1_layer_curves.png")
print(f"  {STEP1_DIR}/fig2_layer_depth.png")
print(f"  {STEP1_DIR}/fig3_early_detection.png")
print(f"  {STEP1_DIR}/fig4_transfer.png")
print(f"  {STEP2_DIR}/fig5_pca_before_after.png")
print(f"  {STEP2_DIR}/fig6_early_detection_contrast.png")
print("="*60)